In [ ]:
# ! pip install scikit-learn scipy matplotlib

In [ ]:
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from models import DynamicPacketGRU

# ---------------------------------------------------------
# 1. Dataset 및 Collate 함수 (패딩 없음)
# ---------------------------------------------------------
class ExactPacketDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        x = torch.tensor(item["x"], dtype=torch.float32)
        label = torch.tensor([item["label"]], dtype=torch.float32)
        return x, label

def collate_fn_no_pad(batch):
    xs = torch.stack([item[0] for item in batch])
    labels = torch.stack([item[1] for item in batch])
    return xs, labels

# ---------------------------------------------------------
# 2. 데이터 불러오기 함수 (전체 데이터 학습용)
# ---------------------------------------------------------
def load_data(original_path):
    all_data = []
    with open(original_path, 'r', encoding='utf-8') as f:
        for line in f:
            all_data.append(json.loads(line))
            
    # 셔플 (데이터 편향 방지 및 재현성을 위해 시드 고정)
    random.seed(42)
    random.shuffle(all_data)
    
    print(f"  [안내] 학습 데이터 개수: {len(all_data)}개")
    return all_data

# ---------------------------------------------------------
# 3. 학습 핵심 함수 (테스트 제거)
# ---------------------------------------------------------
def train_model(model, train_loader, optimizer, criterion, device, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x_batch, label_batch in train_loader:
            x_batch, label_batch = x_batch.to(device), label_batch.to(device)
            batch_size = x_batch.size(0)
            
            direction_idx = torch.ones(batch_size, dtype=torch.long).to(device)
            
            optimizer.zero_grad()
            outputs = model(x_batch, direction_idx=direction_idx, enable_early_exit=False)
            
            final_preds = outputs[:, -1, :] 
            loss = criterion(final_preds, label_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"    Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ---------------------------------------------------------
# 4. 실질적인 순차적(Curriculum) 학습 실행 파트
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DynamicPacketGRU(input_size=18, hidden_size=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

dataset_type = "dctcp"

# 원본 데이터 파일 경로 매핑
dataset_files = {
    3: f"dataset/elephant_dst_to_src/{dataset_type}/seq3/splits_parent_flow/train.jsonl",
    5: f"dataset/elephant_dst_to_src/{dataset_type}/seq5/splits_parent_flow/train.jsonl",
    10: f"dataset/elephant_dst_to_src/{dataset_type}/seq10/splits_parent_flow/train.jsonl"
}

print("=== 커리큘럼 전체 데이터 학습 및 저장 파이프라인 시작 ===")
for seq_len, src_path in dataset_files.items():
    print(f"\n--------------------------------------------------")
    print(f"[Phase] 시퀀스 길이 {seq_len} 단계 시작")
    print(f"--------------------------------------------------")
    
    # 가중치 파일명 정의
    weight_save_path = f"dataset/elephant_dst_to_src/{dataset_type}/seq{seq_len}/weights.pt"
    
    try:
        # 1. 데이터 불러오기 (전체 데이터)
        train_list = load_data(src_path)
        
        # 2. DataLoader 생성
        train_dataset = ExactPacketDataset(train_list)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn_no_pad)
        
        # 3. 모델 학습 (가중치가 누적되며 이어서 학습됨)
        train_model(model, train_loader, optimizer, criterion, device, epochs=5)
        
        # 4. 현재 단계의 가중치 저장
        torch.save(model.state_dict(), weight_save_path)
        print(f"  [성공] 시퀀스 {seq_len} 학습 완료. 가중치가 '{weight_save_path}'에 저장되었습니다.")
        
    except FileNotFoundError:
        print(f"  [오류] {src_path} 파일을 찾을 수 없어 이 단계를 건너넙니다.")

print("\n=== 모든 단계의 학습 및 파일 저장이 완료되었습니다. ===")